# Unstructured Data Processing — Text and Documents

**Course:** Artificial Intelligence · Unit 6 — Language and NLP  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2022). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 24 — Natural Language Processing  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---


## 1. Install Dependencies

In [ ]:
# pip install  # run once in terminal -U spacy

In [ ]:
# pip install  # run once in terminal nltk

In [ ]:
# pip install  # run once in terminal circlify

## 2. Install Spanish Language Model

In [ ]:
#!python3 -m spacy download es

In [ ]:
#import nltk
#nltk.download('stopwords')

In [ ]:
from pathlib import Path

DIR_DATA = Path.cwd() / 'data'
 else None
DIR_DATA

In [ ]:
# pip install  # run once in terminal seaborn

## 3. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter

# Plot libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Text analysis API's
import re
import spacy
import nltk
import unicodedata
from nltk.corpus import stopwords
from nltk import TweetTokenizer
from spacy.lang.es import Spanish

# Preprocessin API´s
from sklearn.preprocessing import LabelEncoder 
from sklearn import preprocessing

# Regularization API´s 
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Feature extration API´s
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Machine Learning Methods 

from sklearn.linear_model import LogisticRegression

# Metrics Libraries
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score


## 4. Initialise spaCy and Load Model

In [ ]:
nlp = spacy.load('es_core_news_sm')

## 5. Load Dataset

TASS-2018: Workshop on Semantic Analysis at SEPLN

In [ ]:
data_raw = pd.read_csv(DIR_DATA + 'TASS2018.csv', sep=';')
data_raw.head(10)

> **Output interpretation:** The dataset head confirms loading was successful. The `content` column holds raw tweet text that will be processed by spaCy for NER and dependency analysis.


## 6. Describe Dataset

In [ ]:
data_raw.info()

> **Output interpretation:** `info()` shows column types and null counts. Drop null `content` rows before NLP processing to prevent errors during tokenisation.


## 7. Content Description 

### 7.1 Tweet Time Series

In [ ]:
data_raw['hour'] = pd.DatetimeIndex(data_raw['date']).hour
data_raw['minute'] = pd.DatetimeIndex(data_raw['date']).minute
fig = px.line(data_raw, x='polarity', y='hour') 
fig.show()

### 7.2 Polarity Distribution 

In [ ]:
tweet_by_polarity = data_raw.groupby("polarity", as_index=False)['content'].count()
tweet_by_polarity.head(10)

In [ ]:
plt.figure(figsize=(15,5))
sns.barplot(data =tweet_by_polarity, x = "polarity", y = "content", alpha=0.8)
plt.title('Polary Frequency by content')
plt.show()

### 7.3 Word Usage in Tweets

In [ ]:
words = {}
for row in tqdm(data_raw['content']):
    doc = nlp(row.lower())
    for token in doc:
        if token.is_alpha and not token.is_stop:
            if token.text in words:
                num_temp = int(words[token.text])
                words[token.text] = num_temp + 1
            else:
                words[token.text] = 1


In [ ]:
df_words = pd.DataFrame([[key, words[key]] for key in words.keys()], columns=['Word', 'Freq'])
df_words.sort_values('Freq')
df_words = df_words[:20]
df_words.head(10)

In [ ]:
def get_colordict(palette,number,start):
    pal = list(sns.color_palette(palette=palette, n_colors=number).as_hex())
    color_d = dict(enumerate(pal, start=start))
    return color_d

In [ ]:
import circlify
# compute circle positions:
circles = circlify.circlify(df_words['Freq'][0:30].tolist(), 
                            show_enclosure=False, 
                            target_enclosure=circlify.Circle(x=0, y=0)
                           )
n = df_words['Freq'][0:30].max()
color_dict = get_colordict('RdYlBu_r',n ,1)

In [ ]:
fig, ax = plt.subplots(figsize=(9,9), facecolor='white')
ax.axis('off')
lim = max(max(abs(circle.x)+circle.r, abs(circle.y)+circle.r,) for circle in circles)
plt.xlim(-lim, lim)
plt.ylim(-lim, lim)

# list of labels
labels = list(df_words['Word'][0:30])
counts = list(df_words['Freq'][0:30])
labels.reverse()
counts.reverse()

# print circles
for circle, label, count in zip(circles, labels, counts):
    x, y, r = circle
    ax.add_patch(plt.Circle((x, y), r, alpha=0.9, color = color_dict.get(count)))
    plt.annotate(label +'\n'+ str(count), (x,y), size=12, va='center', ha='center')
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
pos_freq = {}
for row in tqdm(data_raw['content'].to_list()):
    doc = nlp(row.lower())
    for token in doc:
        if token.pos_ in pos_freq:
            value = pos_freq[token.pos_]
            pos_freq[token.pos_] = value + 1
        else:
            pos_freq[token.pos_] =  1

In [ ]:
df_pos = pd.DataFrame([[key, pos_freq[key]] for key in pos_freq.keys()], columns=['POS', 'Freq'])
df_pos.sort_values('Freq').tail(15)
df_pos = df_pos[:20]
df_pos.head(10)

In [ ]:
plt.figure(figsize=(15,5))
sns.barplot(data = df_pos, x = 'POS', y = 'Freq', alpha=0.8)
plt.title('POS Frequency')
plt.show()

# 8. Preprocesamiento

In [ ]:
def processing(text: str):
    result = ''
    try:
        text = unicodedata.normalize('NFD', text)
        text = text.encode('ascii', 'ignore')
        text_out = text.decode("utf-8")
        text_out = text_out.lower()
        text_out = re.sub("[\U0001f000-\U000e007f]", 'EMOJI', text_out)
        text_out = re.sub(r'(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+'
                          r'|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:\'".,<>?«»“”‘’]))',
                          'URL', text_out)
        text_out = re.sub("@([A-Za-z0-9_]{1,40})", 'MENTION', text_out)
        text_out = re.sub("#([A-Za-z0-9_]{1,40})", 'HASTAG', text_out)
        # Remove patterns
        
        text_out = re.sub(r'\©|\×|\⇔|\_|\»|\«|\~|\#|\$|\€|\Â|\�|\¬', '', text_out)
        text_out = re.sub(r'\,|\;|\:|\!|\¡|\’|\‘|\”|\“|\"|\'|\`', '', text_out)
        text_out = re.sub(r'\}|\{|\[|\]|\(|\)|\<|\>|\?|\¿|\°|\|', '', text_out)
        text_out = re.sub(r'\/|\-|\+|\*|\=|\^|\%|\&|\$', '', text_out)
        text_out = re.sub(r'\b\d+(?:\.\d+)?\s+', '', text_out)
        text_out = re.sub(r'[0-9]', '', text_out)
        text_out = re.sub(r'\s+', ' ', text_out).strip()
        text_out = text_out.rstrip()
        result = text_out if text_out != ' ' else None
    except Exception as e:
        print('Error processing: {0}'.format(e))
    return result

In [ ]:
messages = [processing(row) for row in data_raw['content'].tolist()]
messages

In [ ]:
stop_words = stopwords.words('spanish')
stop_words

In [ ]:
bow = CountVectorizer(analyzer='word', ngram_range=(1, 1), stop_words= stop_words)

In [ ]:
bow.fit(messages)

In [ ]:
x = bow.transform(messages).toarray()
x

In [ ]:
df = pd.DataFrame(x, index=['tweet '+str(i) for i in range(1, 1+len(messages))])
df.round(3)

In [ ]:
y = data_raw['polarity']
y

In [ ]:
Counter(y)

In [ ]:
oversample = SMOTE()
x, y = oversample.fit_resample(x, y)

In [ ]:
Counter(y)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:
softmax = LogisticRegression(multi_class="multinomial", solver="lbfgs")

In [ ]:
softmax.fit(x_train, y_train)

In [ ]:
y_predict = softmax.predict(x_test)
y_predict

In [ ]:
cm = confusion_matrix(y_test, y_predict)
plt.figure()
sns.heatmap(cm , annot =True, linewidth=0.8,fmt=".1f")
plt.title("Softmax")
plt.show()

In [ ]:
print(classification_report(y_test, y_predict))

In [ ]:
f1 = f1_score(y_test, y_predict, average="macro")
precision = precision_score(y_test, y_predict, average="macro")
recall = recall_score(y_test, y_predict, average="macro")
accuracy = accuracy_score(y_test, y_predict, normalize=True)
print('F1: ',f1)
print('Precision: ', precision)
print('Recall: ', recall)
print('Accuracy: ', accuracy)